# Demo del Sistema Inteligente de Alerta Temprana

## Objetivo

Este notebook presenta una demostración interactiva del sistema desarrollado para el proyecto **Sistema Inteligente de Alerta Temprana para Crisis Educativa Municipal**.

A partir de un municipio seleccionado por el usuario, el sistema:

- Consulta los indicadores educativos del municipio.
- Estima su tasa de deserción mediante el modelo Random Forest entrenado.
- Clasifica automáticamente el nivel de riesgo.
- Simula una intervención educativa.
- Estima el impacto potencial de dicha intervención sobre la deserción escolar.

## Carga de librerías

In [84]:
#Librerias
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display

## Carga del proyecto

Se localiza automáticamente la carpeta principal del proyecto para cargar el conjunto de datos y el modelo entrenado.

In [85]:
PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT.name != "educational_crisis_ai":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(PROJECT_ROOT)

/home/juan/Downloads/educational_crisis_ai/educational_crisis_ai


## Carga del conjunto de datos y del modelo

Se carga el conjunto de datos procesado utilizado por el modelo y el modelo Random Forest previamente entrenado.

In [86]:
df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "dataset_modelado.csv"
)

modelo = joblib.load(
    PROJECT_ROOT / "models" / "random_forest_desercion.pkl"
)

print("Dataset:", df.shape)
print("Modelo cargado correctamente")

Dataset: (15707, 85)
Modelo cargado correctamente


## Ejecución de la simulación

In [114]:
def ejecutar_simulacion(b):

    with salida:

        clear_output()

        municipio_demo = selector.value

        escenario = df[
            (df["AÑO"] == 2024) &
            (df["MUNICIPIO"].str.strip().str.upper() == municipio_demo.strip().upper())
        ].copy()

        if escenario.empty:
            print("❌ Municipio no encontrado.")
            return

        escenario = escenario.iloc[[0]]

        print("=" * 60)
        print("MUNICIPIO SELECCIONADO")
        print("=" * 60)

        print(f"Municipio   : {escenario['MUNICIPIO'].values[0]}")
        print(f"Departamento: {escenario['DEPARTAMENTO'].values[0]}")
        print(f"Año         : {escenario['AÑO'].values[0]}")

        print("\nINDICADORES EDUCATIVOS")
        print("-" * 60)

        print(f"Cobertura neta : {escenario['COBERTURA_NETA'].values[0]:.2f}%")
        print(f"Aprobación     : {escenario['APROBACIÓN'].values[0]:.2f}%")
        print(f"Reprobación    : {escenario['REPROBACIÓN'].values[0]:.2f}%")
        print(f"Repitencia     : {escenario['REPITENCIA'].values[0]:.2f}%")
        print(f"Deserción real : {escenario['DESERCIÓN'].values[0]:.2f}%")

        # ==================================================
        # Escenario actual
        # ==================================================

        columnas_excluir = [
            "DESERCIÓN",
            "MUNICIPIO",
            "DEPARTAMENTO",
            "CÓDIGO_MUNICIPIO",
            "DESERCIÓN_ETC",
            "DESERCIÓN_TRANSICIÓN",
            "DESERCIÓN_TRANSICIÓN_ETC",
            "DESERCIÓN_PRIMARIA",
            "DESERCIÓN_PRIMARIA_ETC",
            "DESERCIÓN_SECUNDARIA",
            "DESERCIÓN_SECUNDARIA_ETC",
            "DESERCIÓN_MEDIA",
            "DESERCIÓN_MEDIA_ETC",
            "ETC_ETC"
        ]

        X_actual = escenario.drop(columns=columnas_excluir, errors="ignore")
        X_actual = X_actual[modelo.feature_names_in_]

        pred_actual = modelo.predict(X_actual)[0]

        # ==================================================
        # Simulación de intervención
        # ==================================================

        escenario_intervenido = escenario.copy()

        escenario_intervenido["COBERTURA_NETA"] += cobertura.value
        escenario_intervenido["APROBACIÓN"] += aprobacion.value
        escenario_intervenido["REPROBACIÓN"] += reprobacion.value
        escenario_intervenido["REPITENCIA"] += repitencia.value

        # ==================================================
        # Validaciones
        # ==================================================

        cobertura_final = escenario_intervenido["COBERTURA_NETA"].values[0]
        aprobacion_final = escenario_intervenido["APROBACIÓN"].values[0]
        reprobacion_final = escenario_intervenido["REPROBACIÓN"].values[0]
        repitencia_final = escenario_intervenido["REPITENCIA"].values[0]

        if (
            cobertura_final < 0 or cobertura_final > 100 or
            aprobacion_final < 0 or aprobacion_final > 100 or
            reprobacion_final < 0 or reprobacion_final > 100 or
            repitencia_final < 0 or repitencia_final > 100
        ):
            print("\n❌ Error: los indicadores deben permanecer entre 0% y 100%.")
            return

        if aprobacion_final + reprobacion_final > 100:
            print("\n❌ Error: la suma de Aprobación y Reprobación no puede superar el 100%.")
            return

        # ==================================================
        # Recalcular variables derivadas
        # ==================================================

        escenario_intervenido["BRECHA_COBERTURA"] = (
            escenario_intervenido["COBERTURA_BRUTA"]
            - escenario_intervenido["COBERTURA_NETA"]
        )

        escenario_intervenido["BRECHA_APROBACION"] = (
            escenario_intervenido["APROBACIÓN"]
            - escenario_intervenido["REPROBACIÓN"]
        )

        escenario_intervenido["INDICE_EFICIENCIA"] = (
            escenario_intervenido["APROBACIÓN"] /
            (
                escenario_intervenido["REPROBACIÓN"]
                + escenario_intervenido["REPITENCIA"]
                + escenario_intervenido["DESERCIÓN"]
            )
        )

        X_intervenido = escenario_intervenido.drop(columns=columnas_excluir, errors="ignore")
        X_intervenido = X_intervenido[modelo.feature_names_in_]

        pred_intervenido = modelo.predict(X_intervenido)[0]

        # ==================================================
        # Clasificación del riesgo
        # ==================================================

        def riesgo(valor):
            if valor < 4:
                return "🟢 Bajo"
            elif valor < 8:
                return "🟡 Medio"
            else:
                return "🔴 Alto"

        # ==================================================
        # Mostrar intervención
        # ==================================================

        print("\nINTERVENCIÓN APLICADA")
        print("-" * 60)
        print(f"Cobertura neta : {cobertura.value:+.1f}%")
        print(f"Aprobación     : {aprobacion.value:+.1f}%")
        print(f"Reprobación    : {reprobacion.value:+.1f}%")
        print(f"Repitencia     : {repitencia.value:+.1f}%")

        print("\n" + "=" * 60)
        print("PREDICCIÓN DEL MODELO - TASA DE DESERCIÓN (%)")
        print("=" * 60)

        print(f"Escenario actual      : {pred_actual:.2f}% ({riesgo(pred_actual)})")
        print(f"Con intervención      : {pred_intervenido:.2f}% ({riesgo(pred_intervenido)})")
        print(f"Variación estimada    : {pred_actual - pred_intervenido:.2f} puntos porcentuales")

## Configuración de la intervención

Los controles deslizantes permiten modificar los principales indicadores educativos del municipio seleccionado. Estos cambios representan un escenario hipotético de intervención cuyos efectos serán evaluados por el modelo.

In [116]:
aprobacion = widgets.FloatSlider(
    value=3,
    min=-10,
    max=10,
    step=0.5,
    description="Aprobación"
)

reprobacion = widgets.FloatSlider(
    value=-2,
    min=-10,
    max=10,
    step=0.5,
    description="Reprobación"
)

display(aprobacion)
display(reprobacion)

FloatSlider(value=3.0, description='Aprobación', max=10.0, min=-10.0, step=0.5)

FloatSlider(value=-2.0, description='Reprobación', max=10.0, min=-10.0, step=0.5)

In [115]:
boton.on_click(ejecutar_simulacion)

## Resultados de la simulación

El sistema presenta los indicadores actuales del municipio, estima la tasa de deserción para el escenario actual y compara dicho resultado con el escenario intervenido definido por el usuario.

In [97]:
# Municipios disponibles (2024)
municipios = sorted(
    df.loc[df["AÑO"] == 2024, "MUNICIPIO"]
      .dropna()
      .unique()
)

selector = widgets.Dropdown(
    options=municipios,
    description="Municipio:",
    layout=widgets.Layout(width="450px")
)

boton = widgets.Button(
    description="Ejecutar simulación",
    button_style="success"
)

salida = widgets.Output()

display(selector)
display(boton)
display(salida)

Dropdown(description='Municipio:', layout=Layout(width='450px'), options=('Abejorral', 'Abrego', 'Abriaquí', '…

Button(button_style='success', description='Ejecutar simulación', style=ButtonStyle())

Output()

In [101]:
import pandas as pd

importancias = pd.Series(
    modelo.feature_importances_,
    index=modelo.feature_names_in_
).sort_values(ascending=False)

print(importancias.head(20))

APROBACIÓN                    0.360108
REPROBACIÓN                   0.246943
APROBACIÓN_PRIMARIA           0.239421
REPROBACIÓN_PRIMARIA          0.088778
BRECHA_APROBACION             0.013621
REPROBACIÓN_SECUNDARIA        0.006932
INDICE_EFICIENCIA             0.006453
APROBACIÓN_SECUNDARIA         0.004382
APROBACIÓN_MEDIA              0.003443
APROBACIÓN_TRANSICIÓN_ETC     0.001468
APROBACIÓN_ETC                0.001354
REPROBACIÓN_MEDIA             0.001134
PESO_MUNICIPIO_ETC            0.001096
COBERTURA_BRUTA_TRANSICIÓN    0.001069
COBERTURA_NETA_TRANSICIÓN     0.001057
TASA_MATRICULACIÓN_5_16       0.000987
COBERTURA_NETA_MEDIA          0.000828
COBERTURA_NETA_SECUNDARIA     0.000813
COBERTURA_NETA                0.000799
COBERTURA_NETA_PRIMARIA       0.000757
dtype: float64


In [107]:
print(pd.Series(modelo.feature_importances_,
                index=modelo.feature_names_in_)
      .loc[lambda s: s.index.str.contains("REPITENCIA")])

REPITENCIA                   0.000373
REPITENCIA_TRANSICIÓN        0.000708
REPITENCIA_PRIMARIA          0.000568
REPITENCIA_SECUNDARIA        0.000431
REPITENCIA_MEDIA             0.000479
REPITENCIA_ETC               0.000212
REPITENCIA_TRANSICIÓN_ETC    0.000446
REPITENCIA_PRIMARIA_ETC      0.000259
REPITENCIA_SECUNDARIA_ETC    0.000420
REPITENCIA_MEDIA_ETC         0.000383
dtype: float64


In [108]:
print(pd.Series(modelo.feature_importances_,
                index=modelo.feature_names_in_)
      .loc[lambda s: s.index.str.contains("COBERTURA")])

COBERTURA_NETA                    0.000799
COBERTURA_NETA_TRANSICIÓN         0.001057
COBERTURA_NETA_PRIMARIA           0.000757
COBERTURA_NETA_SECUNDARIA         0.000813
COBERTURA_NETA_MEDIA              0.000828
COBERTURA_BRUTA                   0.000532
COBERTURA_BRUTA_TRANSICIÓN        0.001069
COBERTURA_BRUTA_PRIMARIA          0.000598
COBERTURA_BRUTA_SECUNDARIA        0.000472
COBERTURA_BRUTA_MEDIA             0.000623
COBERTURA_NETA_ETC                0.000248
COBERTURA_NETA_TRANSICIÓN_ETC     0.000464
COBERTURA_NETA_PRIMARIA_ETC       0.000336
COBERTURA_NETA_SECUNDARIA_ETC     0.000364
COBERTURA_NETA_MEDIA_ETC          0.000329
COBERTURA_BRUTA_ETC               0.000330
COBERTURA_BRUTA_TRANSICIÓN_ETC    0.000384
COBERTURA_BRUTA_PRIMARIA_ETC      0.000453
COBERTURA_BRUTA_SECUNDARIA_ETC    0.000260
COBERTURA_BRUTA_MEDIA_ETC         0.000323
BRECHA_COBERTURA                  0.000669
dtype: float64


In [111]:
df[["DESERCIÓN", "APROBACIÓN", "REPROBACIÓN", "REPITENCIA"]].corr()

,DESERCIÓN,APROBACIÓN,REPROBACIÓN,REPITENCIA
DESERCIÓN,1.000000,-0.519418,0.157600,0.089141
APROBACIÓN,-0.519418,1.000000,-0.806975,-0.255620
REPROBACIÓN,0.157600,-0.806975,1.000000,0.321029
REPITENCIA,0.089141,-0.255620,0.321029,1.000000


In [112]:
# Tomamos un municipio de prueba (el que quieras)
escenario = df[
    (df["AÑO"] == 2024) &
    (df["MUNICIPIO"] == "Cartagena")
].iloc[[0]].copy()

# Columnas que no entran al modelo
columnas_excluir = [
    "DESERCIÓN",
    "MUNICIPIO",
    "DEPARTAMENTO",
    "CÓDIGO_MUNICIPIO",
    "DESERCIÓN_ETC",
    "DESERCIÓN_TRANSICIÓN",
    "DESERCIÓN_TRANSICIÓN_ETC",
    "DESERCIÓN_PRIMARIA",
    "DESERCIÓN_PRIMARIA_ETC",
    "DESERCIÓN_SECUNDARIA",
    "DESERCIÓN_SECUNDARIA_ETC",
    "DESERCIÓN_MEDIA",
    "DESERCIÓN_MEDIA_ETC",
    "ETC_ETC"
]

print("Efecto de modificar únicamente la REPROBACIÓN\n")

for r in range(0, 21):

    prueba = escenario.copy()

    prueba["REPROBACIÓN"] = r

    X = prueba.drop(columns=columnas_excluir, errors="ignore")
    X = X.reindex(columns=modelo.feature_names_in_, fill_value=0)

    pred = modelo.predict(X)[0]

    print(f"Reprobación = {r:2d}%  --->  Deserción predicha = {pred:.2f}%")

Efecto de modificar únicamente la REPROBACIÓN

Reprobación =  0%  --->  Deserción predicha = 5.50%
Reprobación =  1%  --->  Deserción predicha = 5.50%
Reprobación =  2%  --->  Deserción predicha = 5.50%
Reprobación =  3%  --->  Deserción predicha = 5.47%
Reprobación =  4%  --->  Deserción predicha = 5.40%
Reprobación =  5%  --->  Deserción predicha = 4.89%
Reprobación =  6%  --->  Deserción predicha = 4.22%
Reprobación =  7%  --->  Deserción predicha = 3.44%
Reprobación =  8%  --->  Deserción predicha = 3.01%
Reprobación =  9%  --->  Deserción predicha = 2.52%
Reprobación = 10%  --->  Deserción predicha = 2.19%
Reprobación = 11%  --->  Deserción predicha = 2.09%
Reprobación = 12%  --->  Deserción predicha = 2.09%
Reprobación = 13%  --->  Deserción predicha = 2.09%
Reprobación = 14%  --->  Deserción predicha = 2.09%
Reprobación = 15%  --->  Deserción predicha = 2.09%
Reprobación = 16%  --->  Deserción predicha = 2.09%
Reprobación = 17%  --->  Deserción predicha = 2.09%
Reprobación = 18%